# Exercice : faire évoluer un modèle de réseau électrique

Aucune connaissance avancée en réseaux électriques n'est nécessaire. Tout ce qu'il faut savoir :

- Le réseau est un **arbre** qui part d'un poste source. Chaque **nœud** consomme et/ou produit de la puissance, en MW.
- La puissance qui traverse une **ligne** est égale à la demande nette (consommation − production) de tout ce qui se
  trouve en aval de cette ligne. Elle peut être **négative** si la production en aval dépasse la consommation : la
  puissance remonte alors vers le poste source.
- Chaque ligne a une **capacité** maximale, quel que soit le sens du flux.

FicFichiers fournis, sous forme de cellules ci-dessous : `network.py` (le modèle), `example.py` (un petit réseau, exécutable), `test_network.py` (tests, lancés avec `pytest`).

## Utilisation du notebook

1. Exécutez d'abord toutes les cellules une fois (*Exécution > Tout exécuter* dans Colab, *Run > Run All Cells* dans Jupyter).
2. Chaque cellule qui commence par `%%writefile` écrit un fichier Python sur le disque. **Pour modifier le code, éditez directement la cellule, puis réexécutez-la** (Maj + Entrée) : sans cela, le fichier n'est pas mis à jour.
3. Relancez ensuite la cellule des tests (`!python -m pytest`).

Vous pouvez ajouter des cellules pour vos essais.

## Étape 1 — Prise en main et correction

Lancez les tests. L'un d'eux échoue. Corrigez le bug.

## Étape 2 — Mise hors service d'une ligne

L'exploitant doit pouvoir simuler la perte d'une ligne (panne ou maintenance). Ajoutez :

- `take_out_of_service(line_name)` et `put_back_in_service(line_name)` ;

Règle métier : un nœud non relié au poste source ne consomme ni ne produit plus rien (les producteurs se déconnectent automatiquement
par sécurité).
Les méthodes déjà existantes doivent rester cohérentes avec l'état du réseau.

Par ailleurs, on veut pouvoir déterminer l'ensemble des nœuds qui ne sont plus reliés au poste source et la consommation totale des nœuds non alimentés.

## Étape 3 — Analyse N-1

Ajoutez `n_minus_1_analysis()` qui, pour chaque ligne en service, simule sa perte et renvoie la puissance non distribuée
correspondante. Le réseau doit être dans le même état après l'appel qu'avant.


## Installation (inutile si pytest est déjà présent)

In [ ]:
%pip install -q pytest

## Le modèle : `network.py`

In [ ]:
%%writefile network.py
"""Simplified model of a radial electrical distribution network.

A radial network is a tree: it starts at a source substation and branches
out towards consumers, without any loop. Each line therefore has an
"upstream" side (towards the source) and a "downstream" side (towards the ends).
"""
from dataclasses import dataclass


@dataclass
class Node:
    name: str
    load_mw: float = 0.0
    generation_mw: float = 0.0  # e.g. solar panels, wind turbine

    @property
    def net_demand_mw(self) -> float:
        """Positive when the node consumes more than it generates."""
        return self.load_mw - self.generation_mw


@dataclass
class Line:
    name: str
    upstream: str    # node on the source side
    downstream: str  # node on the far end side
    capacity_mw: float


class Network:
    def __init__(self, source: str):
        self.source = source
        self.nodes: dict[str, Node] = {source: Node(source)}
        self.lines: dict[str, Line] = {}

    def add_node(self, name, load_mw=0.0, generation_mw=0.0):
        if name in self.nodes:
            raise ValueError(f"Node {name} already exists")
        self.nodes[name] = Node(name, load_mw, generation_mw)

    def add_line(self, name, upstream, downstream, capacity_mw):
        for n in (upstream, downstream):
            if n not in self.nodes:
                raise ValueError(f"Unknown node: {n}")
        self.lines[name] = Line(name, upstream, downstream, capacity_mw)

    def _downstream_lines(self, node):
        return [l for l in self.lines.values() if l.upstream == node]

    def _downstream_demand(self, node):
        """Net demand of the node and of everything downstream of it."""
        total = self.nodes[node].net_demand_mw
        for line in self._downstream_lines(node):
            total += self._downstream_demand(line.downstream)
        return total

    def compute_flows(self):
        """Power flow (MW) in each line. Positive = from upstream to downstream."""
        return {name: self._downstream_demand(l.downstream) for name, l in self.lines.items()}

    def overloaded_lines(self):
        flows = self.compute_flows()
        return [name for name, f in flows.items() if f > self.lines[name].capacity_mw]

    def source_balance_mw(self):
        """Power supplied by the source substation (negative = export upstream)."""
        return self._downstream_demand(self.source)

## Le réseau d'exemple : `example.py`

In [ ]:
%%writefile example.py
"""Example network used by the tests.

    SOURCE --L1--> A --L2--> B
                   |
                   +--L3--> C --L4--> D
"""
from network import Network


def build_example_network():
    net = Network("SOURCE")
    net.add_node("A", load_mw=3.0)
    net.add_node("B", load_mw=2.0)
    net.add_node("C", load_mw=1.0)
    net.add_node("D", load_mw=1.5)
    net.add_line("L1", "SOURCE", "A", capacity_mw=10.0)
    net.add_line("L2", "A", "B", capacity_mw=6.0)
    net.add_line("L3", "A", "C", capacity_mw=4.0)
    net.add_line("L4", "C", "D", capacity_mw=3.0)
    return net


if __name__ == "__main__":
    network = build_example_network()
    for name, f in network.compute_flows().items():
        print(f"{name}: {f:.2f} MW / {network.lines[name].capacity_mw} MW")
    print("Source balance:", network.source_balance_mw(), "MW")
    print("Overloaded lines:", network.overloaded_lines())

In [ ]:
!python example.py

## Les tests : `test_network.py`

In [ ]:
%%writefile test_network.py
import pytest

from example import build_example_network
from network import Network


def test_flows_example_network():
    flows = build_example_network().compute_flows()
    assert flows == pytest.approx({"L1": 7.5, "L2": 2.0, "L3": 2.5, "L4": 1.5})


def test_source_balance():
    assert build_example_network().source_balance_mw() == pytest.approx(7.5)


def test_overload_when_consuming():
    net = build_example_network()
    net.lines["L4"].capacity_mw = 1.0
    assert net.overloaded_lines() == ["L4"]


def test_overload_when_injecting():
    """A solar farm generates far more than the local load: power flows back
    towards the source and exceeds the line capacity."""
    net = Network("SOURCE")
    net.add_node("PV_FARM", load_mw=0.5, generation_mw=8.0)
    net.add_line("L1", "SOURCE", "PV_FARM", capacity_mw=5.0)
    assert net.compute_flows()["L1"] == pytest.approx(-7.5)
    assert net.overloaded_lines() == ["L1"]


def test_unknown_node():
    net = Network("SOURCE")
    with pytest.raises(ValueError):
        net.add_line("L1", "SOURCE", "X", capacity_mw=1.0)

## Lancer les tests

In [ ]:
!python -m pytest -q